# 🧬 EA-F-001 HANA KIM — Production-Grade Flux LoRA Training Pipeline
**Version**: v3.0 Production | **Author**: Gowni | **Date**: June 2026

```
Pipeline Stages:
  [1] Environment Setup & GPU Validation
  [2] Dataset Ingestion & Validation
  [3] Automatic Image Captioning (BLIP-2)
  [4] Train/Validation Split
  [5] LoRA Training Configuration
  [6] Training with Experiment Tracking (W&B + MLflow)
  [7] Training Metrics Dashboard
  [8] Best Checkpoint Selection
  [9] Inference Validation
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STAGE 1: Environment Setup & GPU Validation               ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, os

def install(pkg): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Core deps
packages = [
    'torch torchvision --index-url https://download.pytorch.org/whl/cu121',
    'bitsandbytes', 'transformers', 'accelerate', 'datasets',
    'Pillow', 'numpy', 'matplotlib', 'seaborn', 'pandas',
    'mlflow', 'wandb', 'tqdm', 'pyyaml', 'scikit-learn',
    'lpips', 'clip-anytorch', 'huggingface_hub',
]
print('Installing dependencies...')
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split(), capture_output=True)
print('✅ All dependencies installed')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# GPU Report
print('\n' + '='*60)
print('  GPU ENVIRONMENT REPORT')
print('='*60)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'  ✅ GPU       : {gpu.name}')
    print(f'  ✅ VRAM      : {gpu.total_memory / 1024**3:.1f} GB')
    print(f'  ✅ CUDA      : {torch.version.cuda}')
    print(f'  ✅ PyTorch   : {torch.__version__}')
    vram = gpu.total_memory / 1024**3
    rec = 'A100 recommended' if vram < 40 else 'Excellent for training'
    print(f'  📊 Status    : {rec}')
else:
    print('  ❌ No GPU found! Switch to L4/A10G/A100 in Lightning AI header.')
print('='*60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STAGE 2: Dataset Validation & Health Check                ║
# ╚══════════════════════════════════════════════════════════════╝
import os, glob, hashlib, json
from PIL import Image
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

STUDIO_ROOT   = os.getcwd()
DATASET_DIR   = f'{STUDIO_ROOT}/character_id/train'
REPORTS_DIR   = f'{STUDIO_ROOT}/reports'
os.makedirs(REPORTS_DIR, exist_ok=True)

class DatasetValidator:
    """Comprehensive dataset validation and health reporting."""

    VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    MIN_RESOLUTION   = 512
    MIN_IMAGES       = 30
    MAX_ASPECT_RATIO = 3.0  # width/height or height/width

    def __init__(self, dataset_dir: str):
        self.dataset_dir = Path(dataset_dir)
        self.results = {}

    def _get_all_images(self):
        imgs = []
        for ext in self.VALID_EXTENSIONS:
            imgs += list(self.dataset_dir.glob(f'*{ext}'))
            imgs += list(self.dataset_dir.glob(f'*{ext.upper()}'))
        return imgs

    def _hash_image(self, img_path: Path) -> str:
        with open(img_path, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()

    def validate(self) -> dict:
        images = self._get_all_images()
        captions = list(self.dataset_dir.glob('*.txt'))

        report = {
            'total_images': len(images),
            'total_captions': len(captions),
            'valid': [],
            'issues': defaultdict(list),
            'duplicates': [],
            'resolutions': [],
            'aspect_ratios': [],
            'caption_lengths': [],
            'file_sizes_kb': [],
        }

        hashes = {}
        cap_stems = {p.stem for p in captions}

        for img_path in images:
            stem = img_path.stem
            issues = []

            # Duplicate check
            h = self._hash_image(img_path)
            if h in hashes:
                issues.append(f'DUPLICATE of {hashes[h].name}')
                report['duplicates'].append(img_path.name)
            else:
                hashes[h] = img_path

            # Open image
            try:
                with Image.open(img_path) as img:
                    w, h_img = img.size
                    report['resolutions'].append((w, h_img))
                    ar = max(w, h_img) / min(w, h_img)
                    report['aspect_ratios'].append(ar)

                    if min(w, h_img) < self.MIN_RESOLUTION:
                        issues.append(f'LOW_RES: {w}×{h_img}')
                    if ar > self.MAX_ASPECT_RATIO:
                        issues.append(f'EXTREME_ASPECT: {ar:.2f}')
                    if img.mode not in ('RGB', 'RGBA', 'L'):
                        issues.append(f'UNUSUAL_MODE: {img.mode}')
            except Exception as e:
                issues.append(f'CORRUPT: {e}')

            # File size
            size_kb = img_path.stat().st_size / 1024
            report['file_sizes_kb'].append(size_kb)
            if size_kb < 20:
                issues.append(f'TINY_FILE: {size_kb:.1f}KB')

            # Caption check
            if stem not in cap_stems:
                # Also check stem without extensions like .jpg
                alt_stem = stem.replace('.jpg', '').replace('.jpeg', '').replace('.png', '').replace('.webp', '')
                if alt_stem not in cap_stems:
                    issues.append('MISSING_CAPTION')
            else:
                cap_path = self.dataset_dir / f'{stem}.txt'
                if cap_path.exists():
                    text = cap_path.read_text(encoding='utf-8').strip()
                    report['caption_lengths'].append(len(text))
                    if len(text) < 30:
                        issues.append(f'SHORT_CAPTION: {len(text)} chars')
                    if 'ea_f_001_hana_kim' not in text:
                        issues.append('MISSING_TRIGGER_TOKEN')

            if issues:
                report['issues'][img_path.name] = issues
            else:
                report['valid'].append(img_path.name)

        self.results = report
        return report

    def print_report(self):
        r = self.results
        valid_count = len(r['valid'])
        total = r['total_images']
        health = (valid_count / total * 100) if total > 0 else 0

        grade = '🏆 ELITE' if health >= 90 else ('✅ GOOD' if health >= 70 else ('⚠️ FAIR' if health >= 50 else '❌ POOR'))

        print('\n' + '='*60)
        print('  DATASET HEALTH REPORT')
        print('='*60)
        print(f'  Total Images     : {total}')
        print(f'  Valid Images     : {valid_count} ({health:.1f}%)')
        print(f'  Captions         : {r["total_captions"]}')
        print(f'  Duplicates       : {len(r["duplicates"])}')
        print(f'  Issues Found     : {len(r["issues"])}')
        print(f'  Health Grade     : {grade}')
        if r['resolutions']:
            widths  = [x[0] for x in r['resolutions']]
            heights = [x[1] for x in r['resolutions']]
            print(f'  Avg Resolution   : {np.mean(widths):.0f}×{np.mean(heights):.0f}')
            print(f'  Min Resolution   : {min(widths):.0f}×{min(heights):.0f}')
        if r['file_sizes_kb']:
            print(f'  Avg File Size    : {np.mean(r["file_sizes_kb"]):.0f} KB')
        if r['issues']:
            print('\n  Issues Breakdown:')
            for fname, issues in list(r['issues'].items())[:10]:
                print(f'    ⚠️  {fname}: {" | ".join(issues)}')
            if len(r['issues']) > 10:
                print(f'    ... and {len(r["issues"]) - 10} more')
        print('='*60)

        if total < self.MIN_IMAGES:
            print(f'\n❗ WARNING: Only {total} images found. Minimum {self.MIN_IMAGES} recommended.')
            print(f'   Add {self.MIN_IMAGES - total} more images for professional-quality training.')

    def plot_report(self):
        r = self.results
        if not r['resolutions']:
            print('No data to plot.')
            return

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('Dataset Health Dashboard — EA-F-001 Hana Kim', fontsize=16, fontweight='bold')

        # Resolution scatter
        ws = [x[0] for x in r['resolutions']]
        hs = [x[1] for x in r['resolutions']]
        axes[0,0].scatter(ws, hs, alpha=0.7, c='steelblue', s=80, edgecolors='navy')
        axes[0,0].axhline(512, c='red', ls='--', label='Min 512px')
        axes[0,0].axvline(512, c='red', ls='--')
        axes[0,0].set_xlabel('Width (px)'); axes[0,0].set_ylabel('Height (px)')
        axes[0,0].set_title('Image Resolutions'); axes[0,0].legend()

        # File sizes
        axes[0,1].hist(r['file_sizes_kb'], bins=15, color='mediumseagreen', edgecolor='darkgreen', alpha=0.85)
        axes[0,1].set_xlabel('File Size (KB)'); axes[0,1].set_ylabel('Count')
        axes[0,1].set_title('File Size Distribution')

        # Dataset health pie
        valid = len(r['valid'])
        issues = len(r['issues'])
        dupes = len(r['duplicates'])
        labels = ['Valid', 'Has Issues', 'Duplicates']
        sizes  = [valid, issues, dupes]
        colors = ['#2ecc71', '#e74c3c', '#f39c12']
        axes[1,0].pie([s for s in sizes if s > 0],
                      labels=[l for l, s in zip(labels, sizes) if s > 0],
                      colors=[c for c, s in zip(colors, sizes) if s > 0],
                      autopct='%1.1f%%', startangle=90)
        axes[1,0].set_title('Dataset Quality Breakdown')

        # Caption length dist
        if r['caption_lengths']:
            axes[1,1].hist(r['caption_lengths'], bins=15, color='mediumpurple', edgecolor='purple', alpha=0.85)
            axes[1,1].axvline(100, c='red', ls='--', label='Min 100 chars')
            axes[1,1].set_xlabel('Caption Length (chars)'); axes[1,1].set_ylabel('Count')
            axes[1,1].set_title('Caption Length Distribution'); axes[1,1].legend()

        plt.tight_layout()
        save_path = f'{REPORTS_DIR}/dataset_health.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'\n📊 Dashboard saved: {save_path}')


# RUN VALIDATION
validator = DatasetValidator(DATASET_DIR)
results = validator.validate()
validator.print_report()
validator.plot_report()

# Save JSON report
report_json = {k: v for k, v in results.items() if k not in ('valid', 'resolutions', 'aspect_ratios')}
report_json['issues'] = dict(results['issues'])
with open(f'{REPORTS_DIR}/dataset_validation.json', 'w') as f:
    json.dump(report_json, f, indent=2)
print('📁 Validation report saved to reports/dataset_validation.json')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STAGE 3: Automatic Image Captioning (BLIP-2 + Template)   ║
# ╚══════════════════════════════════════════════════════════════╝
import torch
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import os

STUDIO_ROOT  = os.getcwd()
DATASET_DIR  = f'{STUDIO_ROOT}/character_id/train'
TRIGGER      = 'ea_f_001_hana_kim'
CHAR_BASE    = (
    'Hana Kim, 28-year-old South Korean fashion model, balanced oval face, '
    'deep brown almond eyes, beauty mark below left jawline, dark chestnut hair, '
    'light beige skin, visible pores'
)

# BLIP-2 auto-captioner
class AutoCaptioner:
    def __init__(self, device='cuda'):
        self.device = device
        self.model = None
        self.processor = None

    def load(self):
        from transformers import Blip2Processor, Blip2ForConditionalGeneration
        print('Loading BLIP-2 captioner...')
        self.processor = Blip2Processor.from_pretrained('Salesforce/blip2-opt-2.7b')
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            'Salesforce/blip2-opt-2.7b',
            torch_dtype=torch.float16,
            device_map='auto'
        )
        print('✅ BLIP-2 loaded!')

    def describe(self, img_path: Path) -> str:
        with Image.open(img_path).convert('RGB') as img:
            # Resize for BLIP
            img = img.resize((596, 596), Image.LANCZOS)
            inputs = self.processor(img, return_tensors='pt').to(self.device, torch.float16)
            with torch.no_grad():
                out = self.model.generate(**inputs, max_new_tokens=60)
            return self.processor.decode(out[0], skip_special_tokens=True).strip()

    def build_caption(self, img_path: Path, blip_desc: str) -> str:
        """Combine BLIP description with identity template."""
        # Extract useful keywords from BLIP output
        blip_lower = blip_desc.lower()

        # Detect pose cues
        pose_cues = []
        if any(w in blip_lower for w in ['standing', 'stands']): pose_cues.append('standing')
        if any(w in blip_lower for w in ['sitting', 'sits', 'seated']): pose_cues.append('sitting')
        if any(w in blip_lower for w in ['walking', 'walks']): pose_cues.append('walking')
        if any(w in blip_lower for w in ['profile', 'side']): pose_cues.append('profile view')

        # Detect clothing cues
        clothing_cues = []
        clothing_kws = {
            'dress': 'wearing a dress', 'coat': 'wearing a coat',
            'jacket': 'wearing a jacket', 'blouse': 'wearing a blouse',
            'gown': 'wearing an evening gown', 'knit': 'wearing knitwear',
        }
        for kw, phrase in clothing_kws.items():
            if kw in blip_lower:
                clothing_cues.append(phrase)

        # Detect environment
        env_cues = []
        if any(w in blip_lower for w in ['indoor', 'studio', 'room', 'interior']): env_cues.append('indoor studio')
        if any(w in blip_lower for w in ['outdoor', 'garden', 'park', 'street']): env_cues.append('outdoor setting')

        # Compose final caption
        parts = [TRIGGER, CHAR_BASE]
        if pose_cues: parts.append(', '.join(pose_cues))
        if clothing_cues: parts.append(clothing_cues[0])
        if env_cues: parts.append(env_cues[0])
        parts.append('luxury editorial fashion photography, photorealistic, natural skin texture')

        return ', '.join(parts)


def generate_all_captions(dataset_dir: str, overwrite: bool = False, use_blip: bool = True):
    """Generate captions for all images missing a .txt file."""
    import glob
    dataset_path = Path(dataset_dir)

    VALID_EXTS = ('.jpg', '.jpeg', '.png', '.webp')
    all_images = []
    for ext in VALID_EXTS:
        all_images += list(dataset_path.glob(f'*{ext}'))

    captioner = None
    if use_blip:
        try:
            captioner = AutoCaptioner()
            captioner.load()
        except Exception as e:
            print(f'⚠️  BLIP-2 load failed: {e}. Using template captions.')
            captioner = None

    generated, skipped = 0, 0
    caption_log = []

    for img_path in tqdm(all_images, desc='Captioning images'):
        stem = img_path.stem
        txt_path = img_path.parent / f'{stem}.txt'

        if txt_path.exists() and not overwrite:
            skipped += 1
            continue

        if captioner:
            try:
                blip_desc = captioner.describe(img_path)
                caption = captioner.build_caption(img_path, blip_desc)
            except Exception as e:
                caption = f'{TRIGGER}, {CHAR_BASE}, editorial fashion portrait, photorealistic'
                blip_desc = '(blip failed)'
        else:
            blip_desc = '(template only)'
            caption = f'{TRIGGER}, {CHAR_BASE}, editorial fashion portrait, luxury atmosphere, photorealistic'

        txt_path.write_text(caption, encoding='utf-8')
        caption_log.append({'image': img_path.name, 'blip': blip_desc, 'caption': caption})
        generated += 1

    # Save caption log
    import json
    log_path = Path(STUDIO_ROOT) / 'reports' / 'caption_log.json'
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(caption_log, f, indent=2, ensure_ascii=False)

    print(f'\n✅ Captioning complete:')
    print(f'   Generated : {generated}')
    print(f'   Skipped   : {skipped} (already had captions)')
    print(f'   Log saved : {log_path}')


# RUN CAPTIONING
# Set use_blip=True to use BLIP-2, False for template-only (faster)
generate_all_captions(DATASET_DIR, overwrite=False, use_blip=False)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STAGE 4: Train / Validation Split                         ║
# ╚══════════════════════════════════════════════════════════════╝
import os, shutil, random, glob
from pathlib import Path

STUDIO_ROOT  = os.getcwd()
DATASET_DIR  = f'{STUDIO_ROOT}/character_id/train'
VAL_DIR      = f'{STUDIO_ROOT}/character_id/validation'
VAL_RATIO    = 0.15  # 15% for validation
SEED         = 42

os.makedirs(VAL_DIR, exist_ok=True)
random.seed(SEED)

VALID_EXTS = ('.jpg', '.jpeg', '.png', '.webp')
all_images = []
for ext in VALID_EXTS:
    all_images += [str(p) for p in Path(DATASET_DIR).glob(f'*{ext}')]

random.shuffle(all_images)
n_val = max(1, int(len(all_images) * VAL_RATIO))
val_images   = all_images[:n_val]
train_images = all_images[n_val:]

# Move val images to val folder (copy captions too)
for img_path in val_images:
    img = Path(img_path)
    stem = img.stem
    txt = img.parent / f'{stem}.txt'

    # Copy (not move) to preserve original dataset
    shutil.copy2(img, Path(VAL_DIR) / img.name)
    if txt.exists():
        shutil.copy2(txt, Path(VAL_DIR) / txt.name)

print(f'Dataset Split:')
print(f'  📂 Training   : {len(train_images)} images')
print(f'  📂 Validation : {len(val_images)} images')
print(f'  📂 Val Ratio  : {VAL_RATIO*100:.0f}%')
print(f'  📂 Val Dir    : {VAL_DIR}')
print('✅ Split complete!')

# Save split manifest
import json
split_manifest = {
    'seed': SEED,
    'val_ratio': VAL_RATIO,
    'train_count': len(train_images),
    'val_count': len(val_images),
    'train_images': [Path(p).name for p in train_images],
    'val_images': [Path(p).name for p in val_images],
}
with open(f'{STUDIO_ROOT}/reports/data_split.json', 'w') as f:
    json.dump(split_manifest, f, indent=2)
print('📁 Split manifest saved to reports/data_split.json')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STAGE 5: LoRA Training Configuration (Elite Hyperparams)  ║
# ╚══════════════════════════════════════════════════════════════╝
import os, yaml, json
from pathlib import Path

STUDIO_ROOT  = os.getcwd()
TOOLKIT_DIR  = f'{STUDIO_ROOT}/ai-toolkit'
OUTPUT_DIR   = f'{STUDIO_ROOT}/output'
DATASET_DIR  = f'{STUDIO_ROOT}/character_id'
CONFIG_DIR   = f'{TOOLKIT_DIR}/config'
os.makedirs(CONFIG_DIR, exist_ok=True)

# ── Experiment Hyperparameters ──────────────────────────────────
EXPERIMENT = {
    'name'            : 'ea_f_001_hana_kim_lora_v3',
    'trigger_token'   : 'ea_f_001_hana_kim',
    'lora_rank'       : 32,
    'lora_alpha'      : 16,
    'steps'           : 3000,
    'learning_rate'   : 4e-4,
    'lr_scheduler'    : 'cosine',
    'lr_warmup'       : 200,
    'batch_size'      : 1,
    'grad_accum'      : 2,
    'ema_decay'       : 0.99,
    'max_grad_norm'   : 1.0,
    'dtype'           : 'bf16',
    'save_every'      : 500,
    'sample_every'    : 250,
    'resolutions'     : [512, 768, 1024],
    'optimizer'       : 'adamw8bit',
    'noise_scheduler' : 'flowmatch',  # Flux-specific: not DDPM!
}

# Validation sample prompts (cover all 5 key scenarios)
SAMPLE_PROMPTS = [
    'ea_f_001_hana_kim, Hana Kim, 28-year-old South Korean fashion model, front-facing, '
    'wearing ivory silk blouse, indoor studio, soft Rembrandt lighting, beauty mark below '
    'left jawline, 85mm portrait, photorealistic, ultra-detailed skin',

    'ea_f_001_hana_kim, Hana Kim, 28-year-old South Korean fashion model, 3/4 angle, '
    'wearing tailored camel coat, Parisian outdoor garden, overcast daylight, deep brown '
    'almond eyes, 105mm, editorial luxury',

    'ea_f_001_hana_kim, Hana Kim, 28-year-old South Korean fashion model, full body, '
    'wearing emerald green halterneck maxi dress, luxury marble studio, walking pose, '
    'natural light, visible skin texture, fashion editorial',

    'ea_f_001_hana_kim, Hana Kim, 28-year-old South Korean fashion model, serious expression, '
    'wearing cream sleeveless knit dress, art gallery interior, direct gaze, side lighting, '
    'pores visible, photorealistic',

    'ea_f_001_hana_kim, Hana Kim, 28-year-old South Korean fashion model, profile view right, '
    'wearing black asymmetric evening gown, outdoor golden hour, dark chestnut hair '
    'flyaways, Hasselblad 120mm medium format quality',
]

# Build full ai-toolkit YAML config
config = {
    'job': 'extension',
    'config': {
        'name': EXPERIMENT['name'],
        'process': [{
            'type': 'sd_trainer',
            'training_folder': OUTPUT_DIR,
            'performance_log_every': 100,
            'device': 'cuda:0',
            'trigger_word': EXPERIMENT['trigger_token'],
            'network': {
                'type': 'lora',
                'linear': EXPERIMENT['lora_rank'],
                'linear_alpha': EXPERIMENT['lora_alpha'],
            },
            'save': {
                'dtype': 'float16',
                'save_every': EXPERIMENT['save_every'],
                'max_step_saves_to_keep': 6,
            },
            'datasets': [{
                'folder_path': f'{DATASET_DIR}/train',
                'caption_ext': 'txt',
                'resolution': EXPERIMENT['resolutions'],
                'shuffle_tokens': False,
                'cache_latents_to_disk': True,
                'tag_dropout': 0.0,
            }],
            'train': {
                'batch_size': EXPERIMENT['batch_size'],
                'steps': EXPERIMENT['steps'],
                'gradient_accumulation_steps': EXPERIMENT['grad_accum'],
                'train_unet': True,
                'train_text_encoder': False,
                'gradient_checkpointing': True,
                'noise_scheduler': EXPERIMENT['noise_scheduler'],
                'optimizer': EXPERIMENT['optimizer'],
                'learning_rate': EXPERIMENT['learning_rate'],
                'lr_scheduler': EXPERIMENT['lr_scheduler'],
                'lr_warmup_steps': EXPERIMENT['lr_warmup'],
                'ema_decay': EXPERIMENT['ema_decay'],
                'dtype': EXPERIMENT['dtype'],
                'max_grad_norm': EXPERIMENT['max_grad_norm'],
                'timestep_type': 'weighted',
            },
            'model': {
                'name_or_path': 'black-forest-labs/FLUX.1-dev',
                'is_flux': True,
                'quantize': True,
                'low_vram': True,
            },
            'sample': {
                'sampler': 'flowmatch',
                'sample_every': EXPERIMENT['sample_every'],
                'width': 1024,
                'height': 1024,
                'guidance_scale': 3.5,
                'sample_steps': 28,
                'prompts': SAMPLE_PROMPTS,
                'neg': 'blurry, low quality, distorted, plastic skin, CGI, bad anatomy',
                'seed': 420691337,
            },
        }]
    },
    'meta': {'name': '[name]', 'version': '1.0'}
}

config_path = f'{CONFIG_DIR}/hana_production_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Also save experiment metadata
exp_meta = {**EXPERIMENT, 'config_path': config_path, 'created': str(Path(config_path).stat().st_mtime)}
with open(f'{STUDIO_ROOT}/reports/experiment_config.json', 'w') as f:
    json.dump(exp_meta, f, indent=2)

print('✅ Training configuration written!')
print(f'   Config: {config_path}')
print(f'\n   Hyperparameter Summary:')
for k, v in EXPERIMENT.items():
    print(f'     {k:<20}: {v}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STAGE 6: Experiment Tracking Setup (W&B + MLflow)         ║
# ╚══════════════════════════════════════════════════════════════╝
import os

# ── Weights & Biases ────────────────────────────────────────────
# Get your W&B API key from: https://wandb.ai/authorize
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')  # ← Replace this
WANDB_PROJECT = 'hana-kim-flux-lora'
WANDB_RUN_NAME = f'ea_f_001_rank32_cosine_3k'

os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_PROJECT'] = WANDB_PROJECT

try:
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print('✅ Weights & Biases authenticated!')
    WANDB_ENABLED = True
except Exception as e:
    print(f'⚠️  W&B not available: {e}. Continuing with MLflow only.')
    WANDB_ENABLED = False

# ── MLflow (local) ──────────────────────────────────────────────
import mlflow
MLFLOW_DIR = f'{STUDIO_ROOT}/mlruns'
mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
mlflow.set_experiment('hana_kim_flux_lora')
print(f'✅ MLflow tracking URI: {MLFLOW_DIR}')
print(f'   View with: mlflow ui --backend-store-uri file://{MLFLOW_DIR}')

# ── Start MLflow run ────────────────────────────────────────────
import json
with open(f'{STUDIO_ROOT}/reports/experiment_config.json') as f:
    exp_config = json.load(f)

with mlflow.start_run(run_name=WANDB_RUN_NAME) as run:
    mlflow.log_params({
        'lora_rank'     : exp_config['lora_rank'],
        'lora_alpha'    : exp_config['lora_alpha'],
        'steps'         : exp_config['steps'],
        'learning_rate' : exp_config['learning_rate'],
        'lr_scheduler'  : exp_config['lr_scheduler'],
        'optimizer'     : exp_config['optimizer'],
        'batch_size'    : exp_config['batch_size'],
        'grad_accum'    : exp_config['grad_accum'],
        'ema_decay'     : exp_config['ema_decay'],
        'base_model'    : 'FLUX.1-dev',
        'trigger_token' : exp_config['trigger_token'],
    })
    RUN_ID = run.info.run_id
    print(f'\n✅ MLflow run started: {RUN_ID}')
    print(f'   Experiment: hana_kim_flux_lora')

print('\n📊 Tracking setup complete!')
print('   Metrics will update during training.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STAGE 7: Launch Training + Live Metrics Capture           ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, re, time, mlflow, json
from pathlib import Path

STUDIO_ROOT  = os.getcwd()
TOOLKIT_DIR  = f'{STUDIO_ROOT}/ai-toolkit'
CONFIG_PATH  = f'{TOOLKIT_DIR}/config/hana_production_config.yaml'
LOG_FILE     = f'{STUDIO_ROOT}/reports/training_log.txt'

# Make sure we're in the right dir
import os; os.chdir(TOOLKIT_DIR)

print('🚀 Launching training...')
print(f'   Config: {CONFIG_PATH}')
print('   Live loss will appear every 100 steps.\n')

start_time = time.time()
loss_history = []
lr_history   = []

# Regex patterns to extract metrics from ai-toolkit output
LOSS_PATTERN  = re.compile(r'loss[:\s=]+([0-9.eE+-]+)', re.IGNORECASE)
STEP_PATTERN  = re.compile(r'step[:\s]+([0-9]+)', re.IGNORECASE)
LR_PATTERN    = re.compile(r'lr[:\s=]+([0-9.eE+-]+)', re.IGNORECASE)

mlflow.set_tracking_uri(f'file://{STUDIO_ROOT}/mlruns')
mlflow.set_experiment('hana_kim_flux_lora')

with mlflow.start_run(run_name='training_run', nested=False) as run:
    with open(LOG_FILE, 'w') as log_f:
        proc = subprocess.Popen(
            ['python', 'run.py', CONFIG_PATH],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        step = 0
        for line in proc.stdout:
            print(line, end='')  # Live print
            log_f.write(line)

            # Extract metrics
            step_m = STEP_PATTERN.search(line)
            if step_m: step = int(step_m.group(1))

            loss_m = LOSS_PATTERN.search(line)
            if loss_m and step > 0:
                loss = float(loss_m.group(1))
                loss_history.append({'step': step, 'loss': loss})
                mlflow.log_metric('train_loss', loss, step=step)

            lr_m = LR_PATTERN.search(line)
            if lr_m and step > 0:
                lr = float(lr_m.group(1))
                lr_history.append({'step': step, 'lr': lr})
                mlflow.log_metric('learning_rate', lr, step=step)

        proc.wait()

    elapsed = time.time() - start_time
    mlflow.log_metric('total_training_time_min', elapsed / 60)

    # Save loss history
    metrics = {'loss': loss_history, 'lr': lr_history, 'elapsed_min': elapsed/60}
    metrics_path = f'{STUDIO_ROOT}/reports/training_metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    mlflow.log_artifact(metrics_path)

print(f'\n✅ Training complete in {elapsed/60:.1f} minutes!')
print(f'   Steps logged: {len(loss_history)}')
print(f'   Final loss: {loss_history[-1]["loss"]:.4f}' if loss_history else '   No loss captured')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STAGE 8: Training Metrics Dashboard                       ║
# ╚══════════════════════════════════════════════════════════════╝
import json, numpy as np, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from pathlib import Path
from IPython.display import Image as IPImage, display
import glob

STUDIO_ROOT = os.getcwd()
REPORTS_DIR = f'{STUDIO_ROOT}/reports'

with open(f'{REPORTS_DIR}/training_metrics.json') as f:
    metrics = json.load(f)

steps  = [m['step'] for m in metrics['loss']]
losses = [m['loss'] for m in metrics['loss']]
lr_steps = [m['step'] for m in metrics['lr']]
lrs      = [m['lr']   for m in metrics['lr']]

# Smooth loss with moving average
window = max(1, len(losses) // 20)
smooth_loss = np.convolve(losses, np.ones(window)/window, mode='valid')
smooth_steps = steps[window-1:]

fig = plt.figure(figsize=(18, 12))
fig.suptitle('Training Dashboard — EA-F-001 Hana Kim LoRA', fontsize=18, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.35)

# Loss curve
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(steps, losses, alpha=0.3, color='#3498db', linewidth=0.8, label='Raw loss')
ax1.plot(smooth_steps, smooth_loss, color='#2980b9', linewidth=2.5, label=f'Smoothed (window={window})')
ax1.set_xlabel('Training Step'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Curve', fontweight='bold')
ax1.legend(); ax1.grid(alpha=0.3)
for step_mark in [500, 1000, 1500, 2000, 2500, 3000]:
    if step_mark <= max(steps, default=0):
        ax1.axvline(step_mark, color='gray', ls=':', alpha=0.5)

# LR curve
ax2 = fig.add_subplot(gs[1, 0:2])
if lrs:
    ax2.plot(lr_steps, lrs, color='#e74c3c', linewidth=2)
    ax2.fill_between(lr_steps, lrs, alpha=0.15, color='#e74c3c')
ax2.set_xlabel('Step'); ax2.set_ylabel('Learning Rate')
ax2.set_title('LR Schedule (Cosine Decay)', fontweight='bold')
ax2.grid(alpha=0.3)

# Loss distribution
ax3 = fig.add_subplot(gs[1, 2])
ax3.hist(losses, bins=30, color='#9b59b6', edgecolor='purple', alpha=0.8)
ax3.axvline(np.mean(losses), color='red', ls='--', label=f'Mean: {np.mean(losses):.4f}')
ax3.set_xlabel('Loss Value'); ax3.set_ylabel('Frequency')
ax3.set_title('Loss Distribution', fontweight='bold')
ax3.legend()

# Loss by checkpoint window
ax4 = fig.add_subplot(gs[2, 0:2])
if len(losses) >= 5:
    window_size = len(losses) // 5
    windows = [losses[i*window_size:(i+1)*window_size] for i in range(5)]
    window_labels = ['Steps 1-600', '601-1200', '1201-1800', '1801-2400', '2401-3000']
    window_means = [np.mean(w) for w in windows if w]
    colors_bar = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']
    ax4.bar(window_labels[:len(window_means)], window_means,
            color=colors_bar[:len(window_means)], edgecolor='white', width=0.6)
    ax4.set_ylabel('Average Loss')
    ax4.set_title('Average Loss per Training Phase', fontweight='bold')
    ax4.grid(axis='y', alpha=0.3)
    plt.setp(ax4.xaxis.get_majorticklabels(), rotation=20, ha='right')

# Summary stats box
ax5 = fig.add_subplot(gs[2, 2])
ax5.axis('off')
summary_text = (
    f'Training Summary\n'
    f'───────────────\n'
    f'Total Steps: {max(steps) if steps else 0:,}\n'
    f'Initial Loss: {losses[0]:.4f}\n'
    f'Final Loss: {losses[-1]:.4f}\n'
    f'Best Loss: {min(losses):.4f}\n'
    f'Loss Reduction: {(1 - losses[-1]/losses[0])*100:.1f}%\n'
    f'Training Time: {metrics.get("elapsed_min", 0):.1f} min\n'
    f'LoRA Rank: 32 | Alpha: 16\n'
    f'Optimizer: AdamW8bit'
)
ax5.text(0.1, 0.5, summary_text, transform=ax5.transAxes,
         fontsize=11, verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))

plt.savefig(f'{REPORTS_DIR}/training_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n📊 Training dashboard saved: {REPORTS_DIR}/training_dashboard.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  STAGE 9: Best Model Selection & Export                    ║
# ╚══════════════════════════════════════════════════════════════╝
import os, glob, shutil, json
from pathlib import Path

STUDIO_ROOT = os.getcwd()
OUTPUT_DIR  = f'{STUDIO_ROOT}/output/ea_f_001_hana_kim_lora_v3'
EXPORT_DIR  = f'{STUDIO_ROOT}/DOWNLOAD_ME'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Find all checkpoints
checkpoints = sorted(glob.glob(f'{OUTPUT_DIR}/*.safetensors'))
sample_dirs = sorted(glob.glob(f'{OUTPUT_DIR}/samples/*/'), key=lambda x: int(x.split('/')[-2]) if x.split('/')[-2].isdigit() else 0)

print(f'\n📦 CHECKPOINT CATALOG')
print('='*50)
checkpoint_info = []
for ckpt in checkpoints:
    name = Path(ckpt).name
    size_mb = Path(ckpt).stat().st_size / 1024**2
    step_match = [s for s in ['500','1000','1500','2000','2500','3000'] if s in name]
    step = step_match[0] if step_match else 'final'
    checkpoint_info.append({'name': name, 'step': step, 'size_mb': round(size_mb, 1), 'path': ckpt})
    print(f'  Step {step:>6}: {name} ({size_mb:.1f} MB)')

# Recommend best checkpoint based on training loss
try:
    with open(f'{STUDIO_ROOT}/reports/training_metrics.json') as f:
        metrics = json.load(f)
    losses = [(m['step'], m['loss']) for m in metrics['loss']]

    # Find checkpoint step with lowest loss
    save_steps = [500, 1000, 1500, 2000, 2500, 3000]
    best_step, best_loss = 3000, float('inf')
    for step in save_steps:
        window_losses = [l for s, l in losses if abs(s - step) < 200]
        if window_losses:
            avg = sum(window_losses) / len(window_losses)
            if avg < best_loss:
                best_loss = avg
                best_step = step

    print(f'\n🏆 RECOMMENDED CHECKPOINT: Step {best_step}')
    print(f'   Avg loss near step {best_step}: {best_loss:.4f}')
except Exception as e:
    print(f'\n⚠️  Could not auto-select best checkpoint: {e}')
    best_step = 3000
    print(f'   Defaulting to final step: {best_step}')

# Copy final + best checkpoint to DOWNLOAD_ME
for ckpt in checkpoints:
    dest = Path(EXPORT_DIR) / Path(ckpt).name
    shutil.copy2(ckpt, dest)
    print(f'\n✅ Exported: {Path(ckpt).name}')

# Save catalog
with open(f'{EXPORT_DIR}/checkpoint_catalog.json', 'w') as f:
    json.dump({'checkpoints': checkpoint_info, 'recommended_step': best_step}, f, indent=2)

print(f'\n📥 All files ready in: {EXPORT_DIR}')
print('   Right-click folder in Lightning AI sidebar → Download')
print('\n📦 After download, place in:')
print('   ComfyUI/models/loras/ea_f_001_hana_kim_lora.safetensors')